In [ ]:
"""
Stage 4 — LSTM Model Architecture + Training
Stock Return Prediction | Quant Research Roadmap — Tier 2

Loads sequences from Stage 3 (main.py → load_sequences())
Builds, trains, and saves two models per ticker:
  - Regression  : predicts next-day log return (float)
  - Classifier  : predicts direction (1=up, 0=down)

Both models share the same LSTM backbone — only the output head differs.
Train both and compare: sometimes direction accuracy matters more than
minimising return error, depending on your trading strategy.

Output:
    models/TICKER_regression.keras
    models/TICKER_classifier.keras
    models/TICKER_history.csv       ← loss curves for plotting
"""

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, BatchNormalization, Bidirectional
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# import load_sequences from your main.py
import sys
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from main import load_sequences, NORM_COLS, WINDOW, SEQ_DIR

# ── CONFIG ────────────────────────────────────────────────────────────────────

MODEL_DIR   = "models"
TICKERS     = [
    "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK",
    "HINDUNILVR", "ITC", "SBIN", "BHARTIARTL", "KOTAKBANK",
    "LT", "AXISBANK", "ASIANPAINT", "MARUTI", "TITAN",
]

# Training hyperparameters
BATCH_SIZE  = 32
MAX_EPOCHS  = 100      # EarlyStopping will cut this short
LR          = 1e-3     # Adam initial learning rate
SEED        = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)


# ═══════════════════════════════════════════════════════════════════════════════
# ARCHITECTURE
# ═══════════════════════════════════════════════════════════════════════════════

def build_regression_model(timesteps: int, n_features: int) -> Model:
    """
    Regression LSTM — predicts next-day log return as a float.

    Architecture decisions explained:

    1. Bidirectional LSTM (first layer only):
       Processes the sequence both forward AND backward, giving richer
       context for each timestep. Only on the first layer because stacking
       two bidirectional layers is expensive and rarely helps for daily data.

    2. return_sequences=True on layer 1, False on layer 2:
       Layer 1 needs to pass the full sequence to layer 2 (return_sequences=True).
       Layer 2 only passes its final hidden state to the Dense layers (False).
       This is the standard stacked-LSTM pattern.

    3. Dropout (0.3) after each LSTM:
       Prevents overfitting. Critical here because financial time series
       are noisy and models overfit very easily.

    4. BatchNormalization before Dense layers:
       Stabilises training when return magnitudes vary across stocks.
       Keeps gradients from exploding or vanishing in the dense head.

    5. L2 regularisation on Dense layers (1e-4):
       Additional overfitting control. Small value so it doesn't dominate loss.

    6. Linear output activation:
       We're predicting a continuous return value, not a probability.
       Never use sigmoid/relu on a regression output.

    Shape flow:
        Input          : (batch, 60, 22)
        BiLSTM         : (batch, 60, 128)   ← 64 units × 2 directions
        Dropout        : (batch, 60, 128)
        LSTM           : (batch, 64)         ← final hidden state only
        Dropout        : (batch, 64)
        BatchNorm      : (batch, 64)
        Dense(32, relu): (batch, 32)
        Dense(1, linear): (batch, 1)        ← predicted return
    """
    inputs = Input(shape=(timesteps, n_features), name="ohlcv_sequence")

    # Layer 1 — Bidirectional LSTM
    x = Bidirectional(
        LSTM(64, return_sequences=True, dropout=0.1, recurrent_dropout=0.1),
        name="bilstm_1"
    )(inputs)
    x = Dropout(0.3, name="drop_1")(x)

    # Layer 2 — Unidirectional LSTM (extracts final state)
    x = LSTM(64, return_sequences=False, dropout=0.1, recurrent_dropout=0.1,
             name="lstm_2")(x)
    x = Dropout(0.3, name="drop_2")(x)

    # Dense head
    x = BatchNormalization(name="batch_norm")(x)
    x = Dense(32, activation="relu",
              kernel_regularizer=l2(1e-4), name="dense_1")(x)
    x = Dropout(0.2, name="drop_3")(x)

    outputs = Dense(1, activation="linear", name="return_output")(x)

    model = Model(inputs, outputs, name="lstm_regression")
    model.compile(
        optimizer=Adam(learning_rate=LR, clipnorm=1.0),
        loss="huber",             # more robust than MSE — less sensitive to outlier return days
        metrics=["mae"],
    )
    return model


def build_classifier_model(timesteps: int, n_features: int) -> Model:
    """
    Classification LSTM — predicts direction (1=up, 0=down).

    Same backbone as regression but with a sigmoid output and binary
    cross-entropy loss. We add class_weight in training to handle the
    slight imbalance between up/down days in bull markets.

    Sigmoid output gives a probability — you can tune the threshold
    (default 0.5) to trade off precision vs recall in backtesting.
    E.g. only go long when probability > 0.6 for a more conservative signal.
    """
    inputs = Input(shape=(timesteps, n_features), name="ohlcv_sequence")

    x = Bidirectional(
        LSTM(64, return_sequences=True, dropout=0.1, recurrent_dropout=0.1),
        name="bilstm_1"
    )(inputs)
    x = Dropout(0.3, name="drop_1")(x)

    x = LSTM(64, return_sequences=False, dropout=0.1, recurrent_dropout=0.1,
             name="lstm_2")(x)
    x = Dropout(0.3, name="drop_2")(x)

    x = BatchNormalization(name="batch_norm")(x)
    x = Dense(32, activation="relu",
              kernel_regularizer=l2(1e-4), name="dense_1")(x)
    x = Dropout(0.2, name="drop_3")(x)

    outputs = Dense(1, activation="sigmoid", name="direction_output")(x)

    model = Model(inputs, outputs, name="lstm_classifier")
    model.compile(
        optimizer=Adam(learning_rate=LR, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ═══════════════════════════════════════════════════════════════════════════════
# CALLBACKS
# ═══════════════════════════════════════════════════════════════════════════════

def get_callbacks(model_path: str, monitor: str = "val_loss") -> list:
    """
    Three callbacks that prevent wasted training time and overfitting:

    EarlyStopping:
        Stops training when val_loss hasn't improved for 15 epochs.
        restore_best_weights=True means you get the best checkpoint,
        not the last (overfitted) one.

    ReduceLROnPlateau:
        Halves learning rate when val_loss plateaus for 7 epochs.
        Lets the model fine-tune after the big initial learning phase.
        min_lr=1e-6 prevents the LR from becoming uselessly tiny.

    ModelCheckpoint:
        Saves the best model to disk during training.
        If training crashes at epoch 80, you don't lose everything.
    """
    return [
        EarlyStopping(
            monitor=monitor,
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor=monitor,
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=model_path,
            monitor=monitor,
            save_best_only=True,
            verbose=0,
        ),
    ]


# ═══════════════════════════════════════════════════════════════════════════════
# CLASS WEIGHT (for classifier only)
# ═══════════════════════════════════════════════════════════════════════════════

def compute_class_weights(y_train: np.ndarray) -> dict:
    """
    In a bull market, up days outnumber down days (~55/45).
    Without class weights, the classifier learns to predict 'up' always
    and achieves 55% accuracy trivially — not useful for trading.
    Weighting penalises misclassifying the minority class more heavily.
    """
    n_total  = len(y_train)
    n_up     = y_train.sum()
    n_down   = n_total - n_up
    w_up     = n_total / (2 * n_up)   if n_up   > 0 else 1.0
    w_down   = n_total / (2 * n_down) if n_down > 0 else 1.0
    print(f"    Class weights → up: {w_up:.3f}  down: {w_down:.3f}  "
          f"(up days: {n_up/n_total:.1%})")
    return {1: w_up, 0: w_down}


# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING
# ═══════════════════════════════════════════════════════════════════════════════

def train_regression(X_train, y_train, X_test, y_test, ticker) -> dict:
    print(f"\n  [{ticker}] Training regression model ...")
    os.makedirs(MODEL_DIR, exist_ok=True)

    timesteps, n_features = X_train.shape[1], X_train.shape[2]
    model = build_regression_model(timesteps, n_features)

    if ticker == TICKERS[0]:           # print summary once
        model.summary()

    model_path = os.path.join(MODEL_DIR, f"{ticker}_regression.keras")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks(model_path, monitor="val_loss"),
        verbose=0,                     # suppress per-epoch output; EarlyStopping prints
        shuffle=False,                 # NEVER shuffle time-series batches
    )

    print(f"    Stopped at epoch {len(history.history['loss'])}")
    print(f"    Best val_loss : {min(history.history['val_loss']):.6f}")
    print(f"    Best val_mae  : {min(history.history['val_mae']):.6f}")

    return history.history


def train_classifier(X_train, y_train_dir, X_test, y_test_dir, ticker) -> dict:
    """
    y_train_dir / y_test_dir are direction labels (0 or 1),
    derived from the sign of the return target in Stage 3.
    """
    print(f"\n  [{ticker}] Training classifier model ...")

    timesteps, n_features = X_train.shape[1], X_train.shape[2]
    model = build_classifier_model(timesteps, n_features)

    model_path   = os.path.join(MODEL_DIR, f"{ticker}_classifier.keras")
    class_weights = compute_class_weights(y_train_dir)

    history = model.fit(
        X_train, y_train_dir,
        validation_data=(X_test, y_test_dir),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks(model_path, monitor="val_accuracy"),
        class_weight=class_weights,
        verbose=0,
        shuffle=False,
    )

    print(f"    Stopped at epoch {len(history.history['loss'])}")
    print(f"    Best val_accuracy : {max(history.history['val_accuracy']):.4f}")

    return history.history


def save_history(reg_hist: dict, clf_hist: dict, ticker: str):
    """Saves loss curves to CSV so you can plot them in Stage 5 (evaluation)."""
    max_len = max(len(reg_hist["loss"]), len(clf_hist["loss"]))

    def pad(lst):
        return lst + [np.nan] * (max_len - len(lst))

    df = pd.DataFrame({
        "reg_loss":         pad(reg_hist["loss"]),
        "reg_val_loss":     pad(reg_hist["val_loss"]),
        "reg_mae":          pad(reg_hist["mae"]),
        "reg_val_mae":      pad(reg_hist["val_mae"]),
        "clf_loss":         pad(clf_hist["loss"]),
        "clf_val_loss":     pad(clf_hist["val_loss"]),
        "clf_accuracy":     pad(clf_hist["accuracy"]),
        "clf_val_accuracy": pad(clf_hist["val_accuracy"]),
    })
    path = os.path.join(MODEL_DIR, f"{ticker}_history.csv")
    df.to_csv(path, index_label="epoch")
    print(f"    History saved → {path}")


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD TRAINED MODEL (for Stage 5 — evaluation)
# ═══════════════════════════════════════════════════════════════════════════════

def load_model(ticker: str, model_type: str = "regression") -> Model:
    """
    model_type: 'regression' or 'classifier'

    Usage in Stage 5:
        model = load_model("RELIANCE", "regression")
        y_pred = model.predict(X_test)
    """
    path = os.path.join(MODEL_DIR, f"{ticker}_{model_type}.keras")
    if not os.path.exists(path):
        raise FileNotFoundError(f"No saved model at {path}. Run Stage 4 first.")
    return tf.keras.models.load_model(path)


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("Stage 4 — LSTM Model Training")
    print(f"Tickers : {len(TICKERS)} stocks")
    print(f"Epochs  : up to {MAX_EPOCHS} (EarlyStopping active)")
    print(f"Batch   : {BATCH_SIZE}")
    print("=" * 60)

    # GPU check
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"  GPU detected: {gpus[0].name}")
        tf.config.experimental.set_memory_growth(gpus[0], True)
    else:
        print("  No GPU detected — training on CPU (slower, still works)")

    os.makedirs(MODEL_DIR, exist_ok=True)

    for ticker in TICKERS:
        seq_path = os.path.join(SEQ_DIR, f"{ticker}_X_train.npy")
        if not os.path.exists(seq_path):
            print(f"\n  SKIP {ticker} — sequences not found. Run main.py first.")
            continue

        print(f"\n{'─'*60}")
        print(f"  {ticker}")
        print(f"{'─'*60}")

        # Load sequences from Stage 3
        X_train, y_train, X_test, y_test, train_dates, test_dates = \
            load_sequences(ticker)

        print(f"  X_train: {X_train.shape} | X_test: {X_test.shape}")

        # Direction labels for classifier (sign of return target)
        y_train_dir = (y_train > 0).astype(np.float32)
        y_test_dir  = (y_test  > 0).astype(np.float32)

        # Train both models
        reg_hist = train_regression(X_train, y_train, X_test, y_test, ticker)
        clf_hist = train_classifier(X_train, y_train_dir,
                                    X_test,  y_test_dir, ticker)

        save_history(reg_hist, clf_hist, ticker)

    print("\n" + "=" * 60)
    print("Stage 4 complete.")
    print(f"Models saved to: {MODEL_DIR}/")
    print("\nNext: Stage 5 — Evaluation (Sharpe ratio, directional accuracy)")
    print("Load with:")
    print("  from 04_lstm_model import load_model")
    print("  model = load_model('RELIANCE', 'regression')")
    print("  y_pred = model.predict(X_test)")
    print("=" * 60)